# NYC Homes × NYC Streets — 12 Research Questions (Milestone 1, part 2)

**Course:** Data Engineering and Visualization, Winter 2026 (GIU) · **Team size:** 6 → 12 research questions + 12 visualizations

This notebook answers the team's research questions using **only** the cleaned and integrated tables produced by
[`01_cleaning_integration.ipynb`](01_cleaning_integration.ipynb). Run that notebook first.

| Input file (data/processed/) | One row is | Used by |
|---|---|---|
| `integrated_nta_month.parquet` | one residential neighborhood (NTA 2020) in one month | RQ1, 2, 4, 5, 7–12 |
| `street_month.parquet` | one street (same normalized name in both datasets) in one month | RQ3 |
| `borough_month.csv` | one borough in one month (all rows, incl. permits without a neighborhood) | RQ6 |
| `nta_boundaries.geojson` | one neighborhood shape | RQ5 map |

**How we avoid trivial or misleading answers** (graded: *"avoid simplistic counts"*, *"claims of causation unsupported by the data"*):
1. **Densities, not raw counts.** Big, busy neighborhoods have more of everything. We divide by area (km²) and by years.
2. **Rank correlation (Spearman ρ).** Counts are very skewed (a few neighborhoods are huge), so ranks are more robust than Pearson.
3. **Within-borough comparisons.** Manhattan has both dense permits and dense housing; comparing inside the borough removes that confounder.
4. **Two-way demeaning for timing questions.** Before testing "does street work come *before* complaints?", we remove each
   neighborhood's own average and each month's city-wide average. What is left is "unusual street work here, this month" vs
   "unusual complaints here, this month", so winter and busy places cannot create a fake link.
5. **Out-of-sample testing for prediction** (train 2022–2024, test 2025).
6. Every answer states its **limits**. Correlation is never presented as causation.

All analysis code is in [`src/analysis.py`](../src/analysis.py) (one function per question), so the **same code** feeds this
notebook and the website.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import analysis as A            # our analysis functions (one per research question)

PROCESSED = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'visualizations'            # PNG copies of every chart (for the report / README)
APP_FIG_DIR = ROOT / 'data' / 'app' / 'rq_figures'   # interactive copies used by the website
FIG_DIR.mkdir(exist_ok=True); APP_FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.width', 180)

integrated, street, borough_month, nta_geo = A.load_outputs(PROCESSED)
print('integrated :', integrated.shape, '| months:', integrated['year_month'].nunique(),
      '| neighborhoods:', integrated['nta_code'].nunique())
print('street     :', street.shape)
print('borough    :', borough_month.shape)
print('H6 problems in the integrated table:', f"{integrated['h6_problems'].sum():,}",
      '| T3 permits:', f"{integrated['t3_permits'].sum():,}")


def save(fig, name, width=1100):
    # Save an interactive JSON copy for the website and (if kaleido is installed) a PNG for reports.
    (APP_FIG_DIR / f'{name}.json').write_text(pio.to_json(fig), encoding='utf-8')
    try:
        fig.write_image(FIG_DIR / f'{name}.png', width=width, height=fig.layout.height or 500, scale=2)
    except Exception as err:          # PNG export is optional (needs `pip install kaleido`)
        print(f'(PNG not saved: {err.__class__.__name__})')
    fig.show()


RESULTS = {}   # key numbers collected for the website's story section

integrated : (9456, 119) | months: 48 | neighborhoods: 197
street     : (273696, 14)
borough    : (240, 27)
H6 problems in the integrated table: 3,153,458 | T3 permits: 2,508,047


---
## Member 1 · Takiee-Allah (16001345)

### RQ1. Are neighborhoods with denser street construction also the neighborhoods with denser housing maintenance problems?

**Why it matters:** If the same places carry both burdens, city agencies could coordinate street work and housing inspections there.

**Method:** Totals per neighborhood for 2022-2025, divided by land area and years (per km2 per year). Spearman rank correlation across all residential neighborhoods, then again inside each borough to remove the borough-level confounder.

**Chart choice:** Log-log scatter (one dot per neighborhood, coloured by borough): both densities span several orders of magnitude, so log axes show the spread honestly.

In [2]:
table, fig, st = A.rq1_density_scatter(integrated)
print(f"Spearman rho = {st['rho']:.3f} (p = {st['p']:.2g}, n = {st['n']})")
display(table.round(3))
RESULTS['rq1'] = st
save(fig, 'rq1')

Spearman rho = 0.647 (p = 9.6e-25, n = 197)


,borough,spearman_rho_within_borough
0,BRONX,0.817
1,BROOKLYN,0.192
2,MANHATTAN,-0.776
3,QUEENS,0.768
4,STATEN ISLAND,0.653


**Answer.** Yes, strongly at city level: Spearman ρ = 0.65 across 197 residential neighborhoods (p < 0.001). The places with the most street permits per km² are also the places with the most housing problems per km². The within-borough picture is the real discovery, though. The link is strong in the Bronx (ρ = 0.82), Queens (0.77) and Staten Island (0.65), weak in Brooklyn (0.19), and **reversed in Manhattan (ρ = −0.78)**. Manhattan's most permit-dense areas are commercial cores like Midtown–Times Square (about 7,500 permits/km²/yr) with few tenants filing complaints (about 500 problems/km²/yr). Its complaint hot-spots, Harlem, Hamilton Heights and Washington Heights (about 9,000–10,000 problems/km²/yr), have only about 1,500 permits/km²/yr.

**Limits.** Both densities are driven by how built-up and residential a place is, which is a shared cause and not one causing the other. Density per km² ignores population and the number of housing units, because no population table was joined.

---
## Member 3 · Ziad Khaled Saber Abdullah (17006438)

### RQ2. Do the two domains peak in the same months of the year, or in opposite seasons?

**Why it matters:** Timing matters for staffing: if streets are busiest when complaints are lowest, inspectors and crews compete less for the same months.

**Method:** City-wide monthly totals, each turned into an index (average month = 100) so both fit on one axis. Month-of-year averages identify each peak and low.

**Chart choice:** Two-line time series on a shared index scale with the legal heat season (Oct-May) shaded. A shared index avoids the misleading dual-axis effect.

In [3]:
table, fig, st = A.rq2_seasonal_cycles(integrated)
print({k: v for k, v in st.items()})
display(table.T)
RESULTS['rq2'] = st
save(fig, 'rq2')

{'rho': -0.28473729917498913, 'p': 0.04981703861611117, 'n': 48, 'h6_peak_month': 'Dec', 't3_peak_month': 'Mar', 'h6_low_month': 'Jun', 't3_low_month': 'Feb'}


,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
h6_problems,152.100,109.700,102.900,84.000,70.800,65.500,74.300,73.400,72.600,110.100,129.000,155.600
t3_permits,78.900,72.800,122.500,96.800,100.700,115.800,97.300,103.100,108.800,109.100,90.200,104.200
t3_street_openings,84.500,80.700,122.400,107.000,100.100,114.700,100.700,98.500,106.800,109.500,85.000,90.100
h6_grp_heat_hot_water,229.200,147.000,112.600,72.000,34.500,19.200,17.700,16.000,19.100,110.200,182.900,239.700


**Answer.** They peak in different seasons. Housing problems peak in **December–January** (index about 155 and 152, average month = 100) and bottom out in **June** (66). The swing is driven by heat/hot-water complaints, which are about 15× higher in December than in August. Street permits are much flatter (73–123) and peak in **March** and June, with their low in **February**. Month to month, the two series are slightly negatively related (ρ = −0.28, p = 0.05). Street crews and housing inspectors are busiest at different times of the year.

**Limits.** This is city-wide, so it says nothing about any single neighborhood. Permits measure permission to work, not the days work actually happened.

---
## Member 2 · Kirolous Bassem (16002081)

### RQ3. On the SAME street, are heat and gas complaints higher in months when the street has gas-main work than in months when it does not?

**Why it matters:** Gas work can mean gas service shut-offs, which can take away heat, hot water and cooking gas from the buildings on that street.

**Method:** Street x month table (street names normalized identically in both datasets). Paired comparison: for every street that had months with AND without gas permits, compare its own average complaints in both kinds of months, separately for heat season and summer. Wilcoxon signed-rank test.

**Chart choice:** Grouped bar chart (with vs without gas work, per season): a direct visual of a paired before/after-style comparison.

In [4]:
table, fig, st = A.rq3_same_street_gas(street)
display(table)
RESULTS['rq3'] = st
save(fig, 'rq3')

,season,streets_compared,mean_without_gas_work,mean_with_gas_work,ratio_with_vs_without,share_streets_higher_with_work,wilcoxon_p
0,Heat season (Oct–May),4505,6.908,6.612,0.957,0.361,0.000
1,Summer (Jun–Sep),3962,1.199,1.205,1.005,0.309,0.940


**Answer.** **No.** In 4,505 streets compared with themselves during heat season, heat + gas complaints averaged 6.61 per street-month in months **with** gas-main permits and 6.91 in months **without**, slightly *lower* with work (ratio 0.96). In summer (3,962 streets) it was 1.20 vs 1.20, no difference. The simple story that gas work cuts off heat and causes complaints is not visible at street-month level.

**Limits.** Street names are matched by name, so a long avenue such as Broadway is one "street" across many neighborhoods. A permit does not say whether service was actually shut off. Very short outages may be fixed before tenants complain to HPD.

### RQ4. Does utility street work come BEFORE matching indoor problems (0-3 months later) in the same neighborhood?

**Why it matters:** A lead-lag pattern would let HPD anticipate complaint spikes from permits that DOT has already issued.

**Method:** For 4 work-to-problem pairs and lags 0-3 months: remove each neighborhood average and each month average (two-way demeaning), then Spearman correlation. Fisher 95% confidence intervals.

**Chart choice:** Grouped bar chart of correlation by lag with 95% error bars: shows size, direction and uncertainty together.

In [5]:
table, fig, st = A.rq4_lagged_utility_links(integrated)
display(table.pivot(index='pair', columns='lag_months', values='rho').round(3))
display(table[['pair', 'lag_months', 'rho', 'ci_low', 'ci_high', 'p', 'n']])
RESULTS['rq4'] = st
save(fig, 'rq4')

lag_months,0,1,2,3
pair,,,,
Gas work → gas complaints,-0.001,0.001,0.005,0.003
Gas work → heat/hot-water complaints,0.010,-0.000,-0.002,0.002
Street openings → all complaints,0.015,0.012,0.011,0.012
Water/sewer work → water/plumbing complaints,-0.003,-0.008,-0.022,-0.016


,pair,lag_months,rho,ci_low,ci_high,p,n
0,Water/sewer work → water/plumbing complaints,0,-0.003,-0.023,0.018,0.803,9456
1,Water/sewer work → water/plumbing complaints,1,-0.008,-0.029,0.012,0.424,9259
2,Water/sewer work → water/plumbing complaints,2,-0.022,-0.043,-0.002,0.035,9062
3,Water/sewer work → water/plumbing complaints,3,-0.016,-0.036,0.005,0.143,8865
4,Gas work → heat/hot-water complaints,0,0.010,-0.010,0.030,0.332,9456
5,Gas work → heat/hot-water complaints,1,-0.000,-0.021,0.020,0.990,9259
6,Gas work → heat/hot-water complaints,2,-0.002,-0.023,0.019,0.855,9062
7,Gas work → heat/hot-water complaints,3,0.002,-0.018,0.023,0.825,8865
8,Gas work → gas complaints,0,-0.001,-0.022,0.019,0.892,9456
9,Gas work → gas complaints,1,0.001,-0.019,0.022,0.899,9259


**Answer.** **No timing link.** After removing each neighborhood's own average and each month's city-wide average, every one of the 16 tested correlations (4 work→problem pairs × lags 0–3 months) lies between −0.022 and +0.015. All of them are practically zero, and 15 of 16 are not statistically significant. Unusual utility work in a neighborhood is not followed by unusual matching complaints there.

**Limits.** Monthly granularity could hide effects that last only days. Neighborhoods average many streets, so a very local effect could be diluted. A null result means "no detectable effect at this scale", not "proven zero".

---
## Member 1 · Takiee-Allah (16001345)

### RQ5. Which neighborhoods carry a "double burden": top quartile for BOTH housing problems and street work per km2?

**Why it matters:** These are the places where residents live with both indoor housing problems and outdoor street disruption. That makes them candidates for coordinated action.

**Method:** Neighborhood densities for 2022-2025. Each neighborhood is classed by whether it is in the top 25% for H6, for T3, for both, or for neither. The number of "both" is compared with what independence would give (25% x 25%).

**Chart choice:** Choropleth map of the four classes: geography is the point of the question.

In [6]:
table, fig, st = A.rq5_double_burden_map(integrated, nta_geo)
print('Neighborhood counts per class:', st['counts'])
print('If the two were unrelated we would expect about', st['expected_if_independent'],
      'double-burden neighborhoods (25% x 25% of', st['n'], ')')
print('Double-burden neighborhoods by borough:', st['by_borough'])
display(table.head(25))
RESULTS['rq5'] = st
save(fig, 'rq5')

Neighborhood counts per class: {'Both high (double burden)': 18, 'Housing complaints high only': 32, 'Street work high only': 32, 'Neither in top quartile': 115}
If the two were unrelated we would expect about 12.3 double-burden neighborhoods (25% x 25% of 197 )
Double-burden neighborhoods by borough: {'MANHATTAN': 8, 'BRONX': 6, 'BROOKLYN': 4}


,nta_name,borough,h6_per_km2_year,t3_per_km2_year
65,Fordham Heights,BRONX,"17,391.304","2,290.580"
64,Mount Hope,BRONX,"11,785.805","3,081.992"
62,Mount Eden-Claremont (West),BRONX,"9,886.905","2,119.841"
115,Harlem (South),MANHATTAN,"9,229.478","1,918.097"
121,Inwood,MANHATTAN,"7,710.075","2,783.582"
60,Concourse-Concourse Village,BRONX,"7,418.496","1,797.967"
63,University Heights (South)-Morris Heights,BRONX,"6,874.038","1,763.736"
66,West Farms,BRONX,"4,691.972","2,537.844"
108,Upper West Side-Manhattan Valley,MANHATTAN,"4,665.854","1,757.317"
117,East Harlem (South),MANHATTAN,"3,538.548","2,404.839"


**Answer.** **18 neighborhoods carry a double burden** (top quartile for both housing problems and street permits per km²). If the two were unrelated we would expect about 12, so overlap is about 1.5× chance. They are concentrated in the West/Central Bronx (Fordham Heights, Mount Hope, Mount Eden-Claremont, Concourse, University Heights, West Farms), Upper Manhattan (Harlem South, Inwood and others) and a few Brooklyn areas. On the map these form two connected clusters around the Harlem River.

**Limits.** Quartile cut-offs are a choice: a different threshold changes the count but not the clusters. Density is per area, not per resident.

---
## Member 3 · Ziad Khaled Saber Abdullah (17006438)

### RQ6. How many housing problems are reported per street permit, by borough and month of year?

**Why it matters:** The ratio shows where and when housing distress outweighs street activity, i.e. where the housing side of the story dominates.

**Method:** Borough x month totals (including permits that could not be placed in a neighborhood), pooled over 2022-2025, ratio H6 problems / T3 permits.

**Chart choice:** Annotated heatmap (borough x month): two categorical axes plus one value is exactly what a heatmap is for.

In [7]:
table, fig, st = A.rq6_complaints_per_permit_heatmap(borough_month)
display(table.round(2))
RESULTS['rq6'] = st
save(fig, 'rq6')

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
borough,,,,,,,,,,,,
BRONX,6.570,5.320,2.750,2.880,2.340,1.890,2.430,2.350,2.150,3.140,4.500,4.530
BROOKLYN,2.280,1.790,1.010,1.080,0.860,0.720,0.950,0.910,0.830,1.250,1.650,1.650
MANHATTAN,1.770,1.430,0.870,0.900,0.700,0.550,0.730,0.700,0.650,0.980,1.350,1.390
QUEENS,1.380,1.040,0.550,0.550,0.470,0.380,0.510,0.440,0.420,0.640,1.020,1.160
STATEN ISLAND,0.570,0.390,0.240,0.240,0.280,0.160,0.260,0.290,0.250,0.340,0.400,0.420


**Answer.** The ratio is highest in the **Bronx in winter**: 6.6 housing problems per street permit in January, against 1.9 in June. It is lowest on **Staten Island** (0.16–0.57). Every borough follows the same winter-high, summer-low pattern, because complaints swing with the heat season while permits stay flat (RQ2). The Bronx ratio is about 3× Brooklyn and about 3.5× Manhattan in every month. There, housing distress outweighs street activity far more than anywhere else.

**Limits.** The ratio mixes two different things (complaints and permissions), so it is a comparison index, not a rate. It is pooled over four years.

---
## Member 4 · Ahmed Hany (16004257)

### RQ7. Are housing complaints closed more slowly in neighborhood-months with heavy street work?

**Why it matters:** Street work may make buildings harder to reach or repairs harder to finish (e.g. waiting for utilities).

**Method:** Neighborhood-months with at least 5 problems. Street-work intensity = quartile of permit density compared with the same borough in the same month (z-score), so busy boroughs and seasons do not drive the tiers. Kruskal-Wallis test and Spearman correlation.

**Chart choice:** Box plot by intensity quartile: compares whole distributions of a skewed variable, not just means.

In [8]:
table, fig, st = A.rq7_close_time_by_intensity(integrated)
print(f"Kruskal-Wallis p = {st['kruskal_p']:.3g} | Spearman rho = {st['rho']:.3f} (n = {st['n']:,})")
display(table)
RESULTS['rq7'] = st
save(fig, 'rq7')

Kruskal-Wallis p = 0.000913 | Spearman rho = 0.028 (n = 9,399)


,count,25%,50%,75%
tier,,,,
Q1 lowest,"2,350.000",3.030,4.630,7.270
Q2,"2,350.000",2.980,4.380,6.840
Q3,"2,349.000",3.090,4.710,7.080
Q4 highest,"2,350.000",3.170,4.840,7.380


**Answer.** **Practically no.** The tiers differ statistically (Kruskal–Wallis p = 0.001, n = 9,399 neighborhood-months), but the size is tiny: the median time to close is 4.63 days in the lowest street-work quartile and 4.84 days in the highest. That is about 5 hours, and the correlation is ρ = 0.03. With 9,000+ observations even meaningless differences become "significant". Effect size is what matters here.

**Limits.** The value is the median per neighborhood-month, not per complaint. Closing a complaint depends mostly on landlords and HPD workload, which we do not observe.

### RQ8. Which 15 neighborhoods are under the most combined pressure relative to their own borough?

**Why it matters:** A ranked shortlist is directly actionable for agencies with limited inspectors.

**Method:** Within-borough z-scores of log housing density and log permit density. The stress index is the average of the two.

**Chart choice:** Horizontal grouped bar chart: long neighborhood names stay readable and both components stay visible.

In [9]:
table, fig, st = A.rq8_top_stress_neighborhoods(integrated)
display(table.round(2))
RESULTS['rq8'] = st
save(fig, 'rq8')

,nta_name,borough,z_h6,z_t3,stress_index
124,Astoria (Central),QUEENS,2.030,1.540,1.780
123,Old Astoria-Hallets Point,QUEENS,1.090,2.350,1.720
64,Mount Hope,BRONX,1.210,1.860,1.540
65,Fordham Heights,BRONX,1.550,1.370,1.460
181,St. George-New Brighton,STATEN ISLAND,1.700,1.150,1.430
182,Tompkinsville-Stapleton-Clifton-Fox Hills,STATEN ISLAND,1.410,1.260,1.340
134,Corona,QUEENS,1.390,1.280,1.330
127,Long Island City-Hunters Point,QUEENS,-0.220,2.830,1.300
165,Jamaica,QUEENS,1.240,1.370,1.300
44,Ocean Hill,BROOKLYN,1.060,1.480,1.270


**Answer.** Relative to their own borough, the most pressured neighborhoods are **Astoria (Central)** and **Old Astoria-Hallets Point** in Queens, **Mount Hope** and **Fordham Heights** in the Bronx, and **St. George-New Brighton** and **Tompkinsville-Stapleton** on Staten Island, followed by **Corona** (Queens). Using within-borough z-scores surfaces places a city-wide ranking would hide: Staten Island's North Shore is unremarkable city-wide but stands out strongly inside its borough.

**Limits.** The index weighs both sides equally. Long Island City reaches the top 15 almost only through street work (housing z below 0), so always read both bars and not just the rank.

---
## Member 5 · Abdallah Ahmed Mohamed (16004141)

### RQ9. Is the MIX of street work different in the neighborhoods with the most heat/hot-water complaints?

**Why it matters:** If high-heat-complaint areas have more gas or steam work, utility infrastructure could be a shared underlying factor.

**Method:** Neighborhoods split into 5 equal groups (quintiles) by heat complaints per km2. For each group: share of permits by work kind.

**Chart choice:** 100% stacked bars: compares composition across groups.

In [10]:
table, fig, st = A.rq9_work_mix_by_heat_quintile(integrated)
display(table)
RESULTS['rq9'] = st
save(fig, 'rq9')

,Gas work,Water / sewer work,Steam,Electric / telecom,Paving,Street reconstruction,Building operations,Sidewalk / curb,Occupancy / equipment,Refuse container,Tree work
heat_quintile,,,,,,,,,,,
Q1 fewest,11.800,5.000,0.200,11.900,7.700,7.500,17.900,11.200,0.700,16.500,9.600
Q2,8.100,3.900,0.600,11.500,4.600,5.400,39.600,8.200,1.100,9.600,7.500
Q3,7.600,3.700,0.400,10.100,4.500,3.300,42.500,7.800,1.200,12.300,6.500
Q4,7.000,3.400,0.600,10.600,3.700,3.100,45.800,6.800,1.400,10.600,7.100
Q5 most,7.700,3.200,0.100,8.600,3.700,2.700,47.500,6.700,1.200,10.800,7.800


**Answer.** Yes. The mix shifts from utility work to **building operations**. In the 20% of neighborhoods with the fewest heat complaints per km², building-operation permits are 18% of all street permits. In the 20% with the most, they are **48%**. Over the same range, gas work falls from 12% to 8%, water/sewer from 5% to 3%, paving from 8% to 4% and sidewalk work from 11% to 7%. High-heat-complaint areas are dense residential areas full of building-side work (scaffolds, sheds, renovations), not areas with unusually much gas or steam work.

**Limits.** Work kinds are our grouping of DOT permit types (documented in notebook 01). Shares hide volume: dense neighborhoods have more of every kind in absolute terms.

### RQ10. Did neighborhoods where street work grew the most (first vs last year) also see housing complaints grow?

**Why it matters:** Change-on-change removes everything that is constant about a neighborhood (size, building age, density). That is a stronger test than comparing levels.

**Method:** Percent change from the first to the last full year for both domains, for neighborhoods with a stable baseline (at least 50 problems and 20 permits in the first year). Spearman correlation and a quadrant count.

**Chart choice:** Scatter with quadrant lines at 0%: each quadrant has a plain meaning (both grew, both fell, ...).

In [11]:
table, fig, st = A.rq10_change_on_change(integrated)
print(f"Spearman rho = {st['rho']:.3f} (p = {st['p']:.2g}, n = {st['n']})")
print('Rows: permits grew?  Columns: complaints grew?'); display(st['quadrants'])
RESULTS['rq10'] = st
save(fig, 'rq10')

Spearman rho = -0.008 (p = 0.91, n = 196)
Rows: permits grew?  Columns: complaints grew?


h6_change_pct,False,True
t3_change_pct,,
False,28,60
True,30,78


**Answer.** **No.** Across 196 neighborhoods, the change in street permits from 2022 to 2025 is unrelated to the change in housing problems (ρ = −0.01, p = 0.91). Housing problems grew city-wide (+17%, from 732k to 854k), so most neighborhoods sit above zero whatever happened to their street work. Of the 108 neighborhoods where permits grew, 78 saw complaints grow. Of the 88 where permits fell, 60 still saw complaints grow. That is almost the same share.

**Limits.** A change between two years is noisy for small neighborhoods. A 4-year window may be too short for slower effects such as new construction attracting new residents.

---
## Member 6 · Mohamed Ahmed (16008868)

### RQ11. Which housing metrics move together with which street metrics across neighborhoods?

**Why it matters:** A full matrix shows the whole picture at once and guards against cherry-picking one pair.

**Method:** 12 neighborhood-level metrics (6 per domain). Spearman correlation matrix. The cross-domain block is outlined.

**Chart choice:** Annotated diverging heatmap (-1 to +1): the standard and clearest way to show a correlation matrix.

In [12]:
table, fig, st = A.rq11_correlation_matrix(integrated)
print('Strongest cross-domain pairs:', st['strongest_cross_pairs'])
display(table)
RESULTS['rq11'] = st
save(fig, 'rq11')

Strongest cross-domain pairs: [('H6 problems /km²', 'T3 permits /km²', np.float64(0.65)), ('H6 problems /km²', 'T3 openings /km²', np.float64(0.6)), ('H6 problems /km²', 'T3 new-job share', np.float64(-0.48)), ('H6 problems /km²', 'T3 utility share', np.float64(-0.46)), ('H6 problems /km²', 'T3 emergency share', np.float64(-0.35))]


,H6 problems /km²,H6 heat share,H6 emergency share,H6 violation rate,H6 no-access rate,H6 days to close,T3 permits /km²,T3 openings /km²,T3 utility share,T3 emergency share,T3 new-job share,T3 work duration
H6 problems /km²,1.000,0.180,-0.050,0.500,-0.570,0.020,0.650,0.600,-0.460,-0.350,-0.480,0.330
H6 heat share,0.180,1.000,0.880,0.030,-0.460,-0.630,0.190,0.120,-0.060,0.110,-0.070,-0.130
H6 emergency share,-0.050,0.880,1.000,-0.050,-0.300,-0.630,0.050,0.020,0.090,0.200,0.020,-0.180
H6 violation rate,0.500,0.030,-0.050,1.000,-0.540,-0.170,-0.000,-0.010,-0.240,-0.110,-0.140,0.320
H6 no-access rate,-0.570,-0.460,-0.300,-0.540,1.000,0.350,-0.190,-0.130,0.280,0.200,0.260,-0.260
H6 days to close,0.020,-0.630,-0.630,-0.170,0.350,1.000,0.200,0.180,-0.140,-0.300,-0.200,0.040
T3 permits /km²,0.650,0.190,0.050,-0.000,-0.190,0.200,1.000,0.950,-0.550,-0.540,-0.670,0.240
T3 openings /km²,0.600,0.120,0.020,-0.010,-0.130,0.180,0.950,1.000,-0.420,-0.460,-0.650,0.290
T3 utility share,-0.460,-0.060,0.090,-0.240,0.280,-0.140,-0.550,-0.420,1.000,0.730,0.640,-0.400
T3 emergency share,-0.350,0.110,0.200,-0.110,0.200,-0.300,-0.540,-0.460,0.730,1.000,0.680,-0.450


**Answer.** The strongest cross-domain link is simple density: housing problems/km² vs permits/km² (ρ = 0.65) and vs street openings/km² (0.60). Neighborhoods with dense housing complaints also have a *lower* share of utility (−0.46), emergency (−0.35) and new-job (−0.48) permits. Their street activity is more routine and more renewal-heavy. Inside H6, dense-complaint areas have higher violation rates (0.50) and fewer "no access" inspections (−0.57). Complaints there are more often confirmed, so they reflect real conditions and not just more reporting.

**Limits.** These are correlations between 197 neighborhoods (an ecological analysis). They describe places, not individual buildings or tenants.

### RQ12. Does knowing the last 3 months of street work help PREDICT next month's housing problems in a neighborhood?

**Why it matters:** This is the practical test of the whole project. If street permits carry real signal, HPD could use DOT data for planning.

**Method:** Linear models on log(1+problems): (a) month-of-year only, (b) the neighborhood's own past problems (lags 1, 2, 3, 12) + season, (c) model b + street-work lags, (d) street-work lags + season. Trained on 2022-2024, scored on 2025 (out-of-sample R2 and mean absolute error).

**Chart choice:** Bar chart of out-of-sample R2 per model: one number per model, easy to compare.

In [13]:
table, fig, st = A.rq12_predictive_value(integrated)
print(st)
display(table)
RESULTS['rq12'] = st
save(fig, 'rq12')

{'train_rows': 4728, 'test_rows': 2364, 'test_year': 2025}


,model,test_R2,test_MAE_problems,n_features
0,Seasonality only (month dummies),0.060,270.678,11
1,Housing history (own lags + season),0.924,67.618,15
2,Housing history + street-work lags,0.925,67.527,30
3,Street-work lags + season only,0.131,267.414,26


**Answer.** **Street permits add no predictive power.** A model using only a neighborhood's own complaint history plus season predicts 2025 out-of-sample with **R² = 0.924**. Adding 15 street-work lag features gives **0.925**, an improvement of 0.0001, and mean absolute error barely moves (67.6 → 67.5 problems). Street-work lags plus season alone reach only R² = 0.13. That is the same as "where is it dense", because permits also mark busy places. This is the clearest evidence that the spatial link in RQ1 comes from shared neighborhood characteristics, not a street-to-home mechanism.

**Limits.** These are linear models on log counts. A non-linear model could find small effects, but the gap would have to be large to matter in practice. The test covers one year (2025).

---
## Summary of findings

| RQ | Member | Short answer |
|---|---|---|
| 1 | Takiee-Allah | Strong spatial link (ρ = 0.65), but reversed inside Manhattan (−0.78): commercial cores vs residential uptown |
| 2 | Ziad | Opposite seasons: complaints peak Dec–Jan (heat), permits peak Mar and Jun; ρ = −0.28 |
| 3 | Kirolous | Same street with vs without gas work: no increase in heat/gas complaints (ratio 0.96 in heat season) |
| 4 | Kirolous | No lead–lag link at 0–3 months: all 16 within-neighborhood correlations ≈ 0 |
| 5 | Takiee-Allah | 18 double-burden neighborhoods vs 12 expected; clusters in West Bronx and Upper Manhattan |
| 6 | Ziad | Bronx in January has 6.6 problems per permit; Staten Island under 0.6 all year |
| 7 | Ahmed Hany | Close times differ by about 5 hours between street-work quartiles: significant but irrelevant |
| 8 | Ahmed Hany | Astoria, Mount Hope, Fordham Heights and the Staten Island North Shore top the within-borough index |
| 9 | Abdallah | High-heat-complaint areas: building-operation permits 48% of permits (vs 18%); less utility work |
| 10 | Abdallah | Growth in permits does not predict growth in complaints (ρ = −0.01) |
| 11 | Mohamed | Density is the only strong cross-domain link; dense-complaint areas have more confirmed violations |
| 12 | Mohamed | Adding street-work lags to prediction: R² 0.924 → 0.925, no practical gain |

## Export the story for the website
The website (`app.py`) shows the same 12 charts (saved above as JSON) and the story text below. Writing it from the notebook
keeps the notebook, the website and the report consistent.

In [14]:
story = json.loads((ROOT / 'tools' / 'rq_text.json').read_text(encoding='utf-8'))
(ROOT / 'data' / 'app' / 'story.json').write_text(json.dumps(story, indent=2, ensure_ascii=False), encoding='utf-8')
print('story.json written with', len(story['research_questions']), 'research questions')

story.json written with 12 research questions


## Prompts used (required deliverable)
AI assistance (Claude) was used while building this notebook. The prompts were:
1. *Review our two datasets (H6 Housing Maintenance Code Complaints ygpa-z7cr and T3 Street Construction Permits tqtj-sjs8) against the Milestone 1 requirements and propose an integration key and granularity.*
2. *Write a resumable, paged Socrata download for both datasets for 2022-2025 and a cleaning pipeline (missing values, IQR outliers, inconsistent labels, duplicates, standardized dates/strings/geography).*
3. *Place every street permit in a 2020 Neighborhood Tabulation Area using its WKT geometry (spatial join) and aggregate both datasets to neighborhood x month, then do the post-integration cleaning.*
4. *Propose 12 non-trivial research questions (2 per team member) that connect both domains, with one appropriate chart each, and implement them with honest statistics (densities, Spearman, within-borough, lags, out-of-sample prediction).*
5. *Build a Dash website with dropdown filters, a free-text search mode, a Generate Report button, a map, a cross-domain scatter and a story section, and prepare it for Render deployment.*